# Tutorial: Unsupervised Classification of Agricultural and Forest Areas using CNN

This notebook presents a step-by-step guide on how to use Convolutional Neural Networks (CNN) to automatically identify and classify agricultural and forest areas using Sentinel-2 remote sensing images.

## Contents:
1. Introduction to Remote Sensing for Agriculture and Forestry
2. Data Preparation
3. Model Implementation
4. Training and Validation
5. Results Visualization and Analysis

## 1. Introduction

### 1.1 Remote Sensing in Agriculture and Forestry
Remote sensing is a fundamental tool for agricultural and forest monitoring, enabling:
- Identification of different agricultural crops
- Vegetation development monitoring
- Deforestation detection
- Vegetation health analysis
- Agricultural expansion area identification

### 1.2 Why use CNN?
CNNs are especially effective for agricultural and forest image analysis because:
- They can identify complex cultivation patterns
- They recognize different vegetation growth stages
- They differentiate types of forest cover
- They are robust to seasonal variations

### 1.3 Unsupervised Approach
The unsupervised approach is particularly useful in the agricultural and forest context because:
- It doesn't require extensive labeled data
- It can discover natural land use patterns
- It adapts to different regions and vegetation types

## 2. Environment Setup

First, let's import the necessary libraries:

In [ ]:
import sys
sys.path.append('..')

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from src.data.tiff_loader import TiffLoader
from src.models.unsupervised_cnn import UnsupervisedCNN
import mlflow
import seaborn as sns

# Configure plot style robustly
try:
    plt.style.use('seaborn')
except:
    # Fallback to default style if seaborn is not available
    plt.style.use('default')

%matplotlib inline

## 3. Understanding Sentinel-2 Data for Agriculture and Forestry

### 3.1 Relevant Spectral Bands

The selected bands are especially useful for agricultural and forest analysis:

- **B8 (NIR - 842nm)**:
  - High reflectance in healthy vegetation
  - Excellent for biomass assessment
  - Differentiates crop types

- **B4 (Red - 665nm)**:
  - Chlorophyll absorption
  - Indicates vegetation stress
  - Useful for distinguishing exposed soil

- **B11 (SWIR - 1610nm)**:
  - Sensitive to water content
  - Helps distinguish crop types
  - Useful for identifying irrigated areas

In [ ]:
# Initialize the loader
loader = TiffLoader(
    train_val_dir='../data/tiffs/train',
    prediction_dir='../data/tiffs/predict',
    patch_size=64
)

# Load training and validation data
train_ds, val_ds = loader.load_train_val_data(val_split=0.2)

# Prepare datasets
train_ds = loader.prepare_dataset(train_ds, batch_size=32, shuffle=True, augment=True)
val_ds = loader.prepare_dataset(val_ds, batch_size=32, shuffle=False)

### 3.2 Visualizing Agricultural and Forest Patterns

Let's visualize some examples and learn to identify different patterns:

In [ ]:
from skimage import color

def adjust_image_for_display(img):
    """
    Adjust image for better visualization of agricultural areas
    - Tall vegetation: red
    - Low vegetation: green
    - Bare soil: brown
    """
    # Normalize image
    img_norm = (img - img.min()) / (img.max() - img.min())
    
    # Extract bands
    nir = img_norm[..., 0]  # B8 - NIR
    red = img_norm[..., 1]  # B4 - Red
    swir = img_norm[..., 2]  # B11 - SWIR
    
    # Calculate vegetation index
    veg_index = (nir - red) / (nir + red + 1e-8)
    veg_index = (veg_index + 1) / 2  # Normalize to [0,1]
    
    # Create enhanced color image
    enhanced = np.zeros_like(img_norm)
    
    # Red channel: highlight tall vegetation
    enhanced[..., 0] = veg_index
    
    # Green channel: highlight low vegetation
    enhanced[..., 1] = np.clip(red * (1 - veg_index), 0, 1)
    
    # Blue channel: smooth for contrast
    enhanced[..., 2] = swir * 0.5
    
    # Adjust contrast
    enhanced = np.power(enhanced, 1/1.2)  # gamma correction
    
    # Increase saturation
    hsv = color.rgb2hsv(enhanced)
    hsv[..., 1] *= 1.4  # Increase saturation
    enhanced = color.hsv2rgb(hsv)
    
    return np.clip(enhanced, 0, 1)

def plot_patches(dataset, num_patches=9):
    """Plot a grid of image patches"""
    patches = next(iter(dataset))
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    axes = axes.flatten()
    
    for j, patch in enumerate(patches[:num_patches]):
        enhanced = adjust_image_for_display(patch.numpy())
        axes[j].imshow(enhanced)
        axes[j].axis('off')
        axes[j].set_title(f'Patch {j+1}')
    plt.tight_layout()
    plt.show()

# Plot patches with new color adjustment
plot_patches(train_ds)

## 4. Model Implementation and Training

### 4.1 Model Architecture

Our CNN is optimized to detect:
- Different vegetation types
- Cultivation patterns
- Growth stages
- Agriculture-forest transition areas

In [ ]:
# Initialize the model
model = UnsupervisedCNN(
    input_shape=(64, 64, 3),
    n_clusters=5,  # Number of classes to identify
    latent_dim=128
)

# Train the model
model.train(
    train_ds,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    learning_rate=0.0005,
    dropout_rate=0.3,
    use_data_augmentation=True
)

# 5. Interpretation of Results

## 5.1 Expected Patterns

When analyzing the clusters, look for:

**Agricultural Areas:**
- Regular geometric patterns
- Variations in intensity by cultivation stage
- Well-defined borders between fields

**Forested Areas:**
- Textural patterns more rugged and irregular
- High response in NIR (B8)
- More homogeneous pattern in large areas

**Special Cases:**
- Areas of forest regeneration
- Agroforestry systems
- Agriculture at different stages

In [ ]:
# Get predictions for validation data
val_predictions = model.predict(val_ds)

# Plot some examples with their classifications
def plot_classifications(dataset, predictions, num_samples=5):
    samples = next(iter(dataset))
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
    
    for i in range(num_samples):
        # Original image
        enhanced = adjust_image_for_display(samples[i].numpy())
        axes[0, i].imshow(enhanced)
        axes[0, i].axis('off')
        axes[0, i].set_title('Original')
        
        # Classification
        axes[1, i].imshow(predictions[i], cmap='tab10')
        axes[1, i].axis('off')
        axes[1, i].set_title('Classification')
    
    plt.tight_layout()
    plt.show()

plot_classifications(val_ds, val_predictions)

In [ ]:
# Make predictions
predictions = model.predict(X_train)


# Import necessary modules
from skimage import color
import numpy as np

In [ ]:

# Create figure with more space on the left
plt.figure(figsize=(15, 3*model.n_clusters))
plt.subplots_adjust(left=0.2)

def adjust_image_for_display(img):
    """
    Adjust the image for better visualization of agricultural areas
    - High vegetation: red
    - Exposed soil: green
    - Agriculture: orange
    - Pasture: light green
    """
    # Normalize to [0, 1]
    img_norm = (img - img.min()) / (img.max() - img.min())
    
    # Highlight specific features
    if len(img_norm.shape) == 3:
        # Assuming RGB order
        r, g, b = img_norm[..., 0], img_norm[..., 1], img_norm[..., 2]
        
        # Calculate normalized vegetation index (NDVI-like)
        veg_index = (r - b) / (r + b + 1e-10)
        
        # Create enhanced image
        enhanced = np.zeros_like(img_norm)
        
        # Red channel: highlight high vegetation
        enhanced[..., 0] = np.clip(r * (1 + veg_index), 0, 1)
        
        # Green channel: highlight exposed soil
        enhanced[..., 1] = np.clip(g * (1 - veg_index), 0, 1)
        
        # Blue channel: smooth for contrast
        enhanced[..., 2] = b * 0.
        
        # Adjust contrast
        enhanced = np.power(enhanced, 1/1.2)  # gamma correction
        
        # Increase saturation
        hsv = color.rgb2hsv(enhanced)
        hsv[..., 1] *= 1.4  # Increase saturation
        enhanced = color.hsv2rgb(hsv)
        
        return np.clip(enhanced, 0, 1)
    else:
        return img_norm

for cluster in range(model.n_clusters):
    cluster_samples = X_train[predictions == cluster][:5]
    for j, sample in enumerate(cluster_samples):
        ax = plt.subplot(model.n_clusters, 5, cluster*5 + j + 1)
        
        # Apply adjustments to the image
        adjusted_sample = adjust_image_for_display(sample)
        
        plt.imshow(adjusted_sample)
        #plt.axis('off')
        if j == 0:
            ax.set_ylabel(f'Cluster {cluster}', fontsize=12, rotation=0)
            ax.yaxis.set_label_coords(-0.5, 0.5)

plt.tight_layout(rect=[0.2, 0, 1, 1])
plt.show()

In [ ]:
# Fazer predições
predictions = model.predict(X_train)


# Importar módulos necessários
from skimage import color
import numpy as np

2025-04-05 17:52:55.947940: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 371294208 exceeds 10% of free system memory.


  1/237 ━━━━━━━━━━━━━━━━━━━━ 1:22 348ms/step

2025-04-05 17:52:56.705511: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 33554432 exceeds 10% of free system memory.


237/237 ━━━━━━━━━━━━━━━━━━━━ 27s 114ms/step


In [ ]:


# Criar figura com mais espaço à esquerda
plt.figure(figsize=(15, 3*model.n_clusters))
plt.subplots_adjust(left=0.2)

def adjust_image_for_display(img):
    """
    Ajusta a imagem para melhor visualização de áreas agrícolas
    - Vegetação alta: vermelho
    - Solo exposto: verde
    - Agricultura: alaranjado
    - Pastagem: verde claro
    """
    # Normalizar para [0, 1]
    img_norm = (img - img.min()) / (img.max() - img.min())
    
    # Realçar características específicas
    if len(img_norm.shape) == 3:
        # Assumindo ordem RGB
        r, g, b = img_norm[..., 0], img_norm[..., 1], img_norm[..., 2]
        
        # Calcular índice de vegetação normalizado (NDVI-like)
        veg_index = (r - b) / (r + b + 1e-10)
        
        # Criar imagem realçada
        enhanced = np.zeros_like(img_norm)
        
        # Canal vermelho: realçar vegetação alta
        enhanced[..., 0] = np.clip(r * (1 + veg_index), 0, 1)
        
        # Canal verde: realçar solo exposto
        enhanced[..., 1] = np.clip(g * (1 - veg_index), 0, 1)
        
        # Canal azul: suavizar para dar contraste
        enhanced[..., 2] = b * 0.
        
        # Ajustar contraste
        enhanced = np.power(enhanced, 1/1.2)  # gamma correction
        
        # Aumentar saturação
        hsv = color.rgb2hsv(enhanced)
        hsv[..., 1] *= 1.4  # Aumentar saturação
        enhanced = color.hsv2rgb(hsv)
        
        return np.clip(enhanced, 0, 1)
    else:
        return img_norm

for cluster in range(model.n_clusters):
    cluster_samples = X_train[predictions == cluster][:5]
    for j, sample in enumerate(cluster_samples):
        ax = plt.subplot(model.n_clusters, 5, cluster*5 + j + 1)
        
        # Aplicar ajustes na imagem
        adjusted_sample = adjust_image_for_display(sample)
        
        plt.imshow(adjusted_sample)
        #plt.axis('off')
        if j == 0:
            ax.set_ylabel(f'Cluster {cluster}', fontsize=12, rotation=0)
            ax.yaxis.set_label_coords(-0.5, 0.5)

plt.tight_layout(rect=[0.2, 0, 1, 1])
plt.show()

AttributeError: 'Functional' object has no attribute 'n_clusters'

## 6. Cluster categorization

We need to categorize the clusters with focus on agriculture and forest.

In [ ]:
from src.models.cluster_categorizer import ClusterCategorizer

categorizer = ClusterCategorizer(n_clusters=model.n_clusters)

# Example categories for agriculture and forest
categories_example = [
    ('dense_forest', 'Dense forest'),
    ('established_agriculture', 'Cultures in advanced stage'),
    ('initial_agriculture', 'Areas recently planted or prepared'),
    ('regenerating_forest', 'Areas in forest regeneration process'),
    ('agroforestry', 'Systems that combine agriculture and forestry')
]

# Categorize clusters
for i, (category, description) in enumerate(categories_example):
    if i < model.n_clusters:
        categorizer.categorize_cluster(
            cluster_id=i,
            category=category,
            description=description,
            confidence=0.8
        )

# Visualize results
categorizer.plot_cluster_samples(X_train, predictions)

## 7. Interpretation

### 7.1 Spectral Features

**Forest:**
- High reflectance in NIR (B8)
- Low reflectance in red (B4)
- Intermediate values in SWIR (B11)

**Agriculture:**
- Seasonal variation in reflectance
- Distinct patterns by crop type
- Resposta espectral varia com estágio de crescimento

### 7.2 Spatial Patterns

**Forest:**
- Textura rugosa
- Natural borders
- Gradual density variation

**Agriculture:**
- Geometric shapes
- Well-defined borders
- Visible planting patterns

### 7.3 Validation

To validate your interpretations:
1. Compare with existing soil use maps
2. Verify area history
3. Use local knowledge when possible
4. Consider agricultural seasonality